In [3]:
import torch
from torch.utils.data import DataLoader, random_split, Subset
from pathlib import Path
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor

In [28]:
#data location

ROOT = Path('NWPU-RESISC45')

dataset = ImageFolder(
    root=ROOT,
    transform=ToTensor()
)

print("Dataset location:", ROOT.resolve)
print("Images:", len(dataset))
print("Classes:", len(dataset.classes))
print(dataset.classes)


Dataset location: <bound method Path.resolve of WindowsPath('NWPU-RESISC45')>
Images: 31494
Classes: 1
['NWPU-RESISC45']


In [35]:
#splitting and loading the data

batch_size = 128

dataset_size = len(dataset)

# Split sizes
test_size = int(0.8 * dataset_size)
train_size = int(0.16 * dataset_size)
val_size = dataset_size - train_size - test_size

# Split dataset
train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=4, 
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

print(f"Test set size: {len(test_loader.dataset)}")


Training dataset size: 5039
Validation dataset size: 1260
Test set size: 25195


In [30]:
import torch
import torch.nn as nn
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

class ZeroShotCLIPClassifier(nn.Module):
    def __init__(self, clip_model_name="openai/clip-vit-base-patch32"):
        super(ZeroShotCLIPClassifier, self).__init__()
        self.clip = CLIPModel.from_pretrained(clip_model_name)
        
        # freeze it
        for param in self.clip.parameters():
            param.requires_grad = False
            
    def forward(self, image_inputs, text_classifier_matrix):
        """
        image_inputs: embedding vector(from clið) from the photos
        text_classifier_matrix: vector with all the text vectors
        """
        # Image Embeddings
        image_features = self.clip.get_image_features(**image_inputs)
        
        # Normalize for Cosine Similarity
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        
        # Probabilities for each class
        # [Batch, 512] x [512, 45] -> [Batch, 45]
        similarity_logits = torch.matmul(image_features, text_classifier_matrix)
        
        return similarity_logits


In [31]:
if torch.cuda.is_available():
    device = torch.device("cuda") # NVIDIA Cuda framework
elif torch.backends.mps.is_available():
    device = torch.device("mps") # MacBook MPS framework
else:
    device = torch.device("cpu")
print("Device: ", device)

Device:  cpu


In [8]:
model_id = "openai/clip-vit-base-patch32"

class_names = [
    "airplane", "airport", "baseball_diamond", "basketball_court", "beach",
    "bridge", "chaparral", "church", "circular_farmland", "cloud",
    "commercial_area", "dense_residential", "desert", "forest", "freeway",
    "golf_course", "ground_track_field", "harbor", "industrial_area", "intersection",
    "island", "lake", "meadow", "medium_residential", "mobile_home_park",
    "mountain", "overpass", "palace", "parking_lot", "railway",
    "railway_station", "rectangular_farmland", "river", "roundabout", "runway",
    "sea_ice", "ship", "snowberg", "sparse_residential", "stadium",
    "storage_tank", "tennis_court", "terrace", "thermal_power_station", "wetland"
]
print(len(class_names))

model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)
model.eval()

def get_prompts_for_class(cls_name):
    clean_name = cls_name.replace("_", " ")
    return [
        f"a satellite image of a {clean_name}",
        f"an aerial photograph of {clean_name}",
        f"a top-down view of {clean_name}",
        f"{clean_name} image taken from space"
    ]

45


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

c:\Users\magnu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\magnu\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

c:\Users\magnu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\magnu\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

... (2 outputs left out)

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
text_embeddings = []
with torch.no_grad():
    for cls in class_names:
        prompts = get_prompts_for_class(cls)
        # Breyta texta í form sem CLIP skilur
        inputs = processor(text=prompts, padding=True, return_tensors="pt").to(device)
        raw_outputs = model.get_text_features(**inputs)  # Shape: [4, 512]

        if hasattr(raw_outputs, "text_embeds"):
            outputs = raw_outputs.text_embeds
        elif hasattr(raw_outputs, "pooler_output"):
            outputs = raw_outputs.pooler_output
        else:
            outputs = raw_outputs
        
        # Normalisera og taka meðaltal af promptunum fyrir þennan klasa
        outputs = outputs / outputs.norm(dim=-1, keepdim=True)
        mean_embedding = outputs.mean(dim=0)
        mean_embedding = mean_embedding / mean_embedding.norm(dim=-1) # Endurnormalisera
        text_embeddings.append(mean_embedding)

    # Búum til endanlegt flokkunarfylki: Shape [512, 45]
    text_classifier = torch.stack(text_embeddings).T
print('Size of the text embedding matrix: ' , text_classifier.shape)


size of the text embedding matrix:  torch.Size([512, 45])


In [44]:
correct_predictions = 0
total_images = 0

with torch.no_grad():
    for images, labels in tqdm(test_loader):
        for batch in tqdm(test_loader):
            images = batch[0]   # Fyrsta stakið í listanum eru myndirnar
            labels = batch[1]  # Réttu tölurnar/merkimiðarnir (0 til 44)
        
        # CLIP þarf myndir á PIL formi eða rétt uppsettum tensorum. 
        # Ef 'images' er þegar TorchGeo tensor [Batch, C, H, W], þá getum við breytt því svona:
        # (Ef processorinn kvartar, gætirðu þurft að breyta myndunum aftur í PIL handvirkt í lykkjunni)
        inputs = processor(images=images, return_tensors="pt", do_rescale=False).to(device)


        # Sækjum eiginleika myndanna (Image Embeddings): Shape [Batch, 512]
        raw_image_features = model.get_image_features(**inputs)

        if hasattr(raw_image_features, "text_embeds"):
            image_features = raw_image_features.text_embeds
        elif hasattr(raw_image_features, "pooler_output"):
            image_features = raw_image_features.pooler_output
        else:
            image_features = image_features

        raw_image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # Fylkjafaldvöxtur (Matrix Multiplication) til að fá líkindaskor: [Batch, 512] x [512, 45] -> [Batch, 45]
        similarity_scores = torch.matmul(image_features, text_classifier)
        
        # Finnum hvaða klasi fékk hæsta skorið fyrir hverja mynd
        predictions = similarity_scores.argmax(dim=-1)
        
        # Berum saman við réttu svörin (labels) og teljum árangurinn
        correct_predictions += (predictions == labels.to(device)).sum().item()
        total_images += labels.size(0)

 16%|█▌        | 32/197 [25:57<2:13:48, 48.66s/it]


KeyboardInterrupt: 